#Transformed constructors data
- Read bronze constructors table
- Keep only the columns required for analytics (Drop url column)
- Standardise column names using snake_case (constructorId → constructor_id)
- Rename columns to make them more meaningful (name -> constructor_name)
- Remove duplicate records
- Transform values of columns nationality to Title Case
- Write the transformed data to silver constructors table

In [0]:
%run ../00-common/01.environment.config


In [0]:
%python
bronze_table = f"{catalog_name}.{bronze_schema_name}.constructors"
silver_table = f"{catalog_name}.{silver_schema_name}.constructors"

#Read bronze constructors table

In [0]:
%python
constructors_df = spark.read.table(bronze_table)

In [0]:
%python
display(constructors_df)
       


constructorId,name,nationality,url,ingestion_timestamp,source_file
adams,Adams,american,http://en.wikipedia.org/wiki/Adams_(constructor),2026-06-15T22:33:34.79404Z,dbfs:/Volumes/formula1/landing/files/constructors.json
afm,AFM,german,http://en.wikipedia.org/wiki/Alex_von_Falkenhausen_Motorenbau,2026-06-15T22:33:34.79404Z,dbfs:/Volumes/formula1/landing/files/constructors.json
ags,AGS,french,http://en.wikipedia.org/wiki/Automobiles_Gonfaronnaises_Sportives,2026-06-15T22:33:34.79404Z,dbfs:/Volumes/formula1/landing/files/constructors.json
alfa,Alfa Romeo,swiss,http://en.wikipedia.org/wiki/Alfa_Romeo_in_Formula_One,2026-06-15T22:33:34.79404Z,dbfs:/Volumes/formula1/landing/files/constructors.json
alphatauri,AlphaTauri,italian,http://en.wikipedia.org/wiki/Scuderia_AlphaTauri,2026-06-15T22:33:34.79404Z,dbfs:/Volumes/formula1/landing/files/constructors.json
alpine,Alpine F1 Team,french,http://en.wikipedia.org/wiki/Alpine_F1_Team,2026-06-15T22:33:34.79404Z,dbfs:/Volumes/formula1/landing/files/constructors.json
alta,Alta,british,http://en.wikipedia.org/wiki/Alta_auto_racing_team,2026-06-15T22:33:34.79404Z,dbfs:/Volumes/formula1/landing/files/constructors.json
amon,Amon,new zealander,http://en.wikipedia.org/wiki/Amon_(Formula_One_team),2026-06-15T22:33:34.79404Z,dbfs:/Volumes/formula1/landing/files/constructors.json
apollon,Apollon,swiss,http://en.wikipedia.org/wiki/Apollon_(Formula_One),2026-06-15T22:33:34.79404Z,dbfs:/Volumes/formula1/landing/files/constructors.json
arrows,Arrows,british,http://en.wikipedia.org/wiki/Arrows_Grand_Prix_International,2026-06-15T22:33:34.79404Z,dbfs:/Volumes/formula1/landing/files/constructors.json


#Keep only the columns required for analytics (Drop url column)

In [0]:
%python
 constructors_dropped_df = constructors_df.drop("url")

#Standardise column names using snake_case (constructorId → constructor_id)
#Rename columns to make them more meaningful (name -> constructor_name)

In [0]:
%python
constructors_renamed_df = (constructors_dropped_df.withColumnRenamed("constructorId", "constructor_id")
.withColumnRenamed("name", "constructor_name")
)

#Remove duplicate records

In [0]:
%python
constructors_distinct_df = constructors_renamed_df.dropDuplicates(["constructor_id"])

#Transform values of columns nationality to Title Case

In [0]:
%python
from pyspark.sql import functions as F
constructors_final_df = (constructors_distinct_df
                     .withColumn("nationality", F.initcap(F.col("nationality")))
                   

)

In [0]:
%python
display(constructors_final_df)

constructor_id,constructor_name,nationality,ingestion_timestamp,source_file
adams,Adams,American,2026-06-15T22:33:34.79404Z,dbfs:/Volumes/formula1/landing/files/constructors.json
afm,AFM,German,2026-06-15T22:33:34.79404Z,dbfs:/Volumes/formula1/landing/files/constructors.json
ags,AGS,French,2026-06-15T22:33:34.79404Z,dbfs:/Volumes/formula1/landing/files/constructors.json
alfa,Alfa Romeo,Swiss,2026-06-15T22:33:34.79404Z,dbfs:/Volumes/formula1/landing/files/constructors.json
alphatauri,AlphaTauri,Italian,2026-06-15T22:33:34.79404Z,dbfs:/Volumes/formula1/landing/files/constructors.json
alpine,Alpine F1 Team,French,2026-06-15T22:33:34.79404Z,dbfs:/Volumes/formula1/landing/files/constructors.json
alta,Alta,British,2026-06-15T22:33:34.79404Z,dbfs:/Volumes/formula1/landing/files/constructors.json
amon,Amon,New Zealander,2026-06-15T22:33:34.79404Z,dbfs:/Volumes/formula1/landing/files/constructors.json
apollon,Apollon,Swiss,2026-06-15T22:33:34.79404Z,dbfs:/Volumes/formula1/landing/files/constructors.json
arrows,Arrows,British,2026-06-15T22:33:34.79404Z,dbfs:/Volumes/formula1/landing/files/constructors.json


#Write the transformed data to silver constructors table

In [0]:
%python
(
    constructors_final_df
    .write
    .format("delta") 
    .mode("overwrite")
    .saveAsTable(silver_table)
)

In [0]:
%python
display(spark.table(silver_table))